# YouTube Dubbing System - Colab (GPU) runner

Runs the full pipeline from the repo on a free Colab GPU:
download -> Whisper -> IndicTrans2 -> **Chatterbox voice cloning** -> timeline -> final video.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.
Run the cells top to bottom. If a cell asks to restart the runtime after installing, restart and continue from the next cell.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/harshita10sharma/Youtube-video-dubbing-system.git"

if not os.path.exists("dubbing"):
    !git clone {REPO_URL} dubbing
else:
    !git -C dubbing pull

%cd dubbing
!ls

> **If you haven't pushed the latest code to GitHub yet**, skip the cell above and instead zip the project folder
> (without `data/` and the large notebooks), upload it here and run:
> ```python
> from google.colab import files; files.upload()   # choose dubbing.zip
> !unzip -q -o dubbing.zip -d dubbing
> %cd dubbing
> ```

## 2. Install dependencies

In [ ]:
# Chatterbox first: it pins compatible torch/numpy versions.
!pip install -q chatterbox-tts
!pip install -q yt-dlp faster-whisper deep-translator edge-tts pydub transformers sentencepiece
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit.git
!ffmpeg -version | head -1

## 3. Provide the inputs

* **Reference voice** - a clean 10-30 s WAV of the voice you want the English dub to use.
* **Video** - either upload a file, or set `VIDEO_URL` (YouTube sometimes blocks Colab IPs; uploading always works).

In [ ]:
from google.colab import files
import shutil, os

os.makedirs("data/audio", exist_ok=True)
os.makedirs("data/videos", exist_ok=True)

print("Upload the reference voice WAV:")
up = files.upload()
REFERENCE = "data/audio/reference.wav"
shutil.move(next(iter(up)), REFERENCE)

VIDEO_URL = ""   # e.g. "https://www.youtube.com/watch?v=XXXX"  (leave empty to upload a file)

if VIDEO_URL:
    SOURCE_ARGS = f'--url "{VIDEO_URL}"'
else:
    print("Upload the video file:")
    up = files.upload()
    VIDEO = "data/videos/source.mp4"
    shutil.move(next(iter(up)), VIDEO)
    SOURCE_ARGS = f"--video {VIDEO}"

print("Ready:", SOURCE_ARGS)

## 4. Run the pipeline
Takes roughly 10-25 minutes for a 15 minute video on a T4. Re-running with `--resume` continues where it stopped.

In [ ]:
!python app.py {SOURCE_ARGS} --tts-backend chatterbox --reference-voice {REFERENCE} --resume

## 5. Quality check

In [ ]:
# Timing summary printed by the merger, and any errors.
!grep -E "Timing report|ERROR|Successfully placed|Detected" logs/pipeline.log | tail -20

In [ ]:
# Listen to a few 10 s excerpts of the dubbed audio track.
import torchaudio
from IPython.display import Audio, display

audio, sr = torchaudio.load("data/audio/dubbed_audio.wav")
for t in [10, 60, 180, 300]:
    if t * sr >= audio.shape[1]:
        break
    clip = audio[:, int(t * sr): int((t + 10) * sr)]
    torchaudio.save(f"check_{t}.wav", clip, sr)
    print(f"--- around {t}s ---")
    display(Audio(f"check_{t}.wav"))

In [ ]:
import json
segs = json.load(open("data/transcripts/translated.json", encoding="utf-8"))
for s in segs[:5]:
    print(f"[{s['start']:.1f}-{s['end']:.1f}s]
  original: {s['text']}
  english : {s['translated']}
")

## 6. Download the result

In [ ]:
from google.colab import files
files.download("data/output/final_dubbed_video.mp4")
# Optional: also keep the transcripts and logs
# !zip -qr outputs.zip data/transcripts logs && files.download("outputs.zip")